# Análisis Exploratorio y Selección de Instancias: PACE 2017 Track B

## Contexto del Estudio
Este notebook tiene como objetivo realizar la selección de instancias de prueba para la evaluación del **Algoritmo Evolutivo (AE)** desarrollado para el problema de *Minimum Fill-in* (Cordalización de Grafos).

Para asegurar la validez científica y la comparabilidad de los resultados, se utilizan grafos provenientes del **PACE 2017 Challenge (Track B)**, una competencia internacional dedicada específicamente a este problema. A diferencia de los grafos generados aleatoriamente, estas instancias representan estructuras reales y desafíos topológicos estandarizados por la comunidad científica.

## Objetivos del Notebook
1.  **Escaneo de Metadatos:** Leer las propiedades topológicas básicas (Número de Nodos $|V|$, Número de Aristas $|E|$ y Densidad $D$) de todo el conjunto de datos sin cargar la estructura completa en memoria.
2.  **Filtrado de Viabilidad:** Descartar instancias triviales (demasiado pequeñas) o computacionalmente intratables para una implementación en Python puro (instancias masivas de >1,000 nodos).
3.  **Análisis de Distribución:** Visualizar la dispersión de los grafos en función de su tamaño y densidad para identificar clústeres representativos.
4.  **Selección Estratégica:** Seleccionar un subconjunto de **9 instancias** que cubran distintos grados de dificultad, siguiendo las directrices de diseño experimental de Eiben & Smith (2015):
    * **3 Instancias Pequeñas (Small):** Validación funcional y convergencia rápida.
    * **3 Instancias Medianas (Medium):** Análisis de robustez paramétrica.
    * **3 Instancias Grandes (Large):** Prueba de estrés y escalabilidad.

## Criterios de Selección
Dado que el problema de *Minimum Fill-in* es NP-Completo, el tiempo de cómputo crece exponencialmente con el tamaño del grafo. Para esta tesis de maestría, se definen los siguientes rangos de interés:

| Categoría | Rango de Nodos ($N$) | Propósito Experimental |
| :--- | :--- | :--- |
| **Small** | $20 \le N \le 60$ | Depuración, Sanity Check y validación de optimalidad. |
| **Medium** | $61 \le N \le 150$ | **Núcleo del estudio.** Balance entre complejidad y tiempo de ejecución. |
| **Large** | $151 \le N \le 400$ | Límite superior de la implementación actual. Evaluación de crecimiento del AES (Average Evaluations to Solution). |

## Origen de los Datos
* **Fuente:** [PACE 2017 Track B Repository](https://pacechallenge.org/2017/minimum-fill-in/)
* **Formato:** DIMACS (`.graph`)
* **Métrica Objetivo:** *Minimum Fill-in* (Número de aristas añadidas para volver al grafo cordal).

---
*Nota: La selección busca maximizar la diversidad topológica dentro de cada categoría (ej. seleccionar grafos dispersos y densos dentro del mismo rango de tamaño).*

## 1. Carga de Instancias y Generación de Línea Base (Baseline)

En esta primera etapa, procesamos el conjunto de datos del benchmark **PACE 2017 Track B (Minimum Fill-in)**.
El objetivo es transformar los archivos de texto crudo (*Raw Edge Lists*) en un estructura tabular que permita seleccionar las instancias más adecuadas para la experimentación.

**Metodología de Procesamiento:**
1.  **Lectura y Normalización:** Se cargan los grafos ignorando comentarios y headers no estándar. Se mapean las etiquetas originales de los nodos (que pueden ser cadenas de texto) a enteros consecutivos $0 \dots N-1$ para optimizar el rendimiento en `networkx`.
2.  **Cálculo de Métricas Topológicas:**
    * **Tamaño:** Nodos ($|V|$) y Aristas ($|E|$).
    * **Densidad:** Calculada como $\rho = \frac{2|E|}{|V|(|V|-1)}$.
3.  **Evaluación de Dificultad (Heurística):** Se ejecuta el algoritmo **Greedy Minimum Degree (MD)** (importado desde `utils`) para obtener un costo de *fill-in* de referencia.
    * *Interpretación:* El valor `MinDegree_FillIn` actúa como un indicador de la complejidad estructural del grafo. Instancias con un alto *fill-in* heurístico representan mejores candidatos para probar la eficacia del Algoritmo Genético frente a métodos constructivos tradicionales.

In [8]:
import os
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm
from utils.heuristics import greedy_minimum_degree

# Configuración visual de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PACE_DIR = "../data/raw/PACE2017B"


def load_pace_graph(filepath):
    """
    Carga grafo desde archivo sin header, ignorando comentarios.
    Convierte etiquetas a enteros consecutivos.
    """
    G = nx.Graph()
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.split()
            u, v = parts[0], parts[1]
            G.add_edge(u, v)

    # Convertir nodos a int 0..N-1 para velocidad
    G = nx.convert_node_labels_to_integers(G)
    return G

def get_category(n):
    if n <= 60: return 'Small'
    if n <= 150: return 'Medium'
    return 'Large'

def process_graph_metrics(filepath):
    """Procesa un solo archivo y calcula métricas + heurística."""
    filename = os.path.basename(filepath)
    G = load_pace_graph(filepath)

    if G is None or G.number_of_nodes() == 0:
        return None

    n = G.number_of_nodes()
    m = G.number_of_edges()
    density = (2 * m) / (n * (n - 1)) if n > 1 else 0

    # Calcular Heurística solo si el grafo es viable (ej. < 1000 nodos)
    # para no bloquear la máquina si hay un grafo monstruoso.
    if n <= 1000:
        _, heuristic_val = greedy_minimum_degree(G, compute_cost=True)
    else:
        heuristic_val = -2 # Código para "Demasiado grande para calcular rápido"

    return {
        "Filename": filename,
        "Nodes": n,
        "Edges": m,
        "Density": density,
        "MinDegree_FillIn": heuristic_val,
        "Category": get_category(n)
    }

# --- EJECUCIÓN PRINCIPAL ---

print(f"Escaneando directorio: {PACE_DIR}")
files = [f for f in os.listdir(PACE_DIR) if f.endswith('.graph')]
print(f"Procesando {len(files)} archivos...")

data = []
# Barra de carga
for f in tqdm(files):
    path = os.path.join(PACE_DIR, f)
    row = process_graph_metrics(path)
    if row:
        data.append(row)

df = pd.DataFrame(data)

# Ordenar y mostrar
df = df.sort_values(by="Nodes").reset_index(drop=True)
print("Procesamiento completado.")
display(df.head(10))

Escaneando directorio: ../data/raw/PACE2017B
Procesando 100 archivos...


  0%|          | 0/100 [00:00<?, ?it/s]

Procesamiento completado.


,Filename,Nodes,Edges,Density,MinDegree_FillIn,Category
0,43.graph,27,69,0.1966,0,Small
1,9.graph,27,254,0.7236,9,Small
2,74.graph,30,307,0.7057,0,Small
3,50.graph,30,349,0.8023,4,Small
4,75.graph,40,539,0.6910,3,Small
5,51.graph,40,587,0.7526,5,Small
6,20.graph,50,59,0.0482,2,Small
7,42.graph,66,1460,0.6807,12,Medium
8,76.graph,72,1121,0.4386,25,Medium
9,24.graph,75,158,0.0569,65,Medium


## 2. Categorización y Selección Visual

En esta etapa filtramos las instancias para mantenernos dentro del rango viable para una tesis de maestría ($20 \le N \le 400$) y las clasificamos en tres grupos de dificultad.

Se utiliza un gráfico interactivo para seleccionar manualmente 9 instancias (3 por categoría).
**Criterio de Selección:** Se buscan grafos que representen un desafío para la heurística *Minimum Degree* (burbujas grandes) y que cubran diferentes densidades.

* **Eje X:** Tamaño del grafo (Nodos).
* **Eje Y:** Densidad.
* **Tamaño de la Burbuja:** Costo de la heurística (`MinDegree_FillIn`). A mayor tamaño, mayor dificultad estructural detectada.

In [9]:
import plotly.express as px

# --- 1. FILTRADO Y CATEGORIZACIÓN ---

MIN_NODES = 20
MAX_NODES = 400

print(f"Filtrando grafos entre {MIN_NODES} y {MAX_NODES} nodos...")
df_viable = df[(df['Nodes'] >= MIN_NODES) & (df['Nodes'] <= MAX_NODES)].copy()

df_viable['Category'] = df_viable['Nodes'].apply(get_category)

# --- 2. LÓGICA DE ESCALADO VISUAL (EL ARREGLO) ---

# Definimos el tamaño mínimo y máximo en PÍXELES que queremos ver en pantalla
MIN_BUBBLE_SIZE = 8   # Tamaño mínimo (para que los pequeños se vean)
MAX_BUBBLE_SIZE = 50  # Tamaño máximo (para los difíciles)

# Obtenemos los valores reales min y max de tus datos
val_min = df_viable['MinDegree_FillIn'].min()
val_max = df_viable['MinDegree_FillIn'].max()

# Función de normalización (Min-Max Scaling)
def scale_size(val):
    # Si todos tienen el mismo valor, devolver el tamaño base
    if val_max == val_min:
        return MIN_BUBBLE_SIZE + 5

    # Fórmula: Base + (Proporción * Rango_Visual)
    normalized = (val - val_min) / (val_max - val_min)
    return MIN_BUBBLE_SIZE + (normalized * (MAX_BUBBLE_SIZE - MIN_BUBBLE_SIZE))

df_viable['Visual_Size'] = df_viable['MinDegree_FillIn'].apply(scale_size)

# --- 3. VISUALIZACIÓN INTERACTIVA ---

fig = px.scatter(
    df_viable,
    x="Nodes",
    y="Density",
    color="Category",
    size="Visual_Size",
    hover_data={
        'Filename': True,
        'Nodes': True,
        'Density': ':.4f',
        'MinDegree_FillIn': True, # Mostramos el valor REAL en el tooltip
        'Visual_Size': False,     # Ocultamos el valor trucado
        'Category': False
    },
    title="<b>Mapa de Selección PACE 2017</b><br>Tamaño = Dificultad (Escalado)",
    color_discrete_map={
        "Small": "#2ca02c",  # Verde
        "Medium": "#1f77b4", # Azul
        "Large": "#d62728"   # Rojo
    },
    labels={"Nodes": "Número de Nodos", "Density": "Densidad del Grafo"}
)

fig.update_layout(
    template="plotly_white",
    height=600,
    legend_title_text='Categoría'
)

fig.show()

# --- 4. TABLA DE AYUDA PARA SELECCIÓN ---
print("Top 5 más difíciles (mayor Fill-in) por categoría:")
for cat in ['Small', 'Medium', 'Large']:
    subset = df_viable[df_viable['Category'] == cat]
    top_hard = subset.sort_values(by='MinDegree_FillIn', ascending=False).head(5)
    print(f"\n--- {cat} ---")
    display(top_hard[['Filename', 'Nodes', 'Density', 'MinDegree_FillIn']])

Filtrando grafos entre 20 y 400 nodos...


Top 5 más difíciles (mayor Fill-in) por categoría:

--- Small ---


,Filename,Nodes,Density,MinDegree_FillIn
1,9.graph,27,0.7236,9
5,51.graph,40,0.7526,5
3,50.graph,30,0.8023,4
4,75.graph,40,0.6910,3
6,20.graph,50,0.0482,2



--- Medium ---


,Filename,Nodes,Density,MinDegree_FillIn
31,40.graph,147,0.6806,377
17,3.graph,101,0.1663,325
24,26.graph,120,0.6868,254
30,89.graph,138,0.6705,235
29,92.graph,132,0.0295,212



--- Large ---


,Filename,Nodes,Density,MinDegree_FillIn
75,71.graph,301,0.2568,8133
74,72.graph,301,0.1220,4740
68,56.graph,276,0.5494,4465
67,65.graph,276,0.0197,4322
78,94.graph,357,0.0980,3349


## 3. Selección Final y Exportación del Portfolio

Basado en el análisis visual y cuantitativo anterior, se seleccionan manualmente 9 instancias que conforman el "Portfolio de Prueba".

**Criterios de Inclusión:**
1.  **Representatividad:** Un balance entre instancias de baja y alta densidad.
2.  **Desafío:** Prioridad a instancias con un alto `MinDegree_FillIn` (burbujas grandes), ya que indican estructuras topológicas donde las heurísticas voraces son sub-óptimas.
3.  **Diversidad de Tamaño:** 3 instancias por categoría (Small, Medium, Large).

El conjunto seleccionado se exporta a `data/processed/selected_instances.csv` para ser consumido por el módulo de experimentación (`run_sensitivity.py`).

In [10]:
# --- 1. SELECCIÓN MANUAL ---
# 1 Fácil, 1 Medio, 1 Difícil (o interesante) por categoría.

MY_SELECTION = [
    # --- SMALL (20-60 nodos) ---
    "20.graph",     # Fácil
    "51.graph",     # Medio
    "9.graph",      # Difícil

    # --- MEDIUM (61-150 nodos) ---
    "42.graph",     # Fácil
    "3.graph",      # Medio, Densidad de 0.16 pero fill-in de 325
    "40.graph",     # Difícil, Denso

    # --- LARGE (151-400 nodos) ---
    "28.graph",     # Fácil
    "65.graph",     # Medio, Densidad de 0.01 pero fill-in de 4,322
    "71.graph"      # Difícil, Fill-in masivo de 8,133
]

# --- 2. FILTRADO Y VALIDACIÓN ---

print(f"Buscando {len(MY_SELECTION)} instancias seleccionadas...")

# Filtramos el DataFrame original para quedarnos solo con los elegidos
df_final = df[df['Filename'].isin(MY_SELECTION)].copy()

# Ordenamos por tamaño para mantener el orden lógico
df_final = df_final.sort_values(by='Nodes')

# Verificación de integridad
if len(df_final) != len(MY_SELECTION):
    missing = set(MY_SELECTION) - set(df_final['Filename'])
    print(f"No se encontraron {len(missing)} archivos en la base de datos:")
    print(missing)
else:
    print("Todas las instancias fueron encontradas exitosamente.\n")

# --- 3. EXPORTACIÓN ---

OUTPUT_PATH = "../data/processed/selected_instances.csv"
# Asegurar que el directorio existe
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Guardamos las columnas clave.
cols_to_save = ['Filename', 'Category', 'Nodes', 'Edges', 'Density', 'MinDegree_FillIn']
df_final[cols_to_save].to_csv(OUTPUT_PATH, index=False)

print(f"Selección guardada en: {OUTPUT_PATH}")

# Validación de diversidad
print("\nDistribución por Categoría:")
print(df_final['Category'].value_counts())

Buscando 9 instancias seleccionadas...
Todas las instancias fueron encontradas exitosamente.

Selección guardada en: ../data/processed/selected_instances.csv

Distribución por Categoría:
Category
Small     3
Medium    3
Large     3
Name: count, dtype: int64
